<a href="https://colab.research.google.com/github/s251198/CS521_das44_HW3/blob/main/multi_norm_adversarial_training_cifar10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS 521 HW3, Problem 3: Multi-Norm Adversarial Training

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

All the code is in this notebook. Complete each place that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the evaluation or the default hyperparameters,
unless the homework tells you to change them.

**Threat model.** For each test image $x$, the perturbation region is
$B_\infty(x, 8/255) \cup B_2(x, 0.5) \cup B_1(x, 12)$.
The union accuracy is the percentage of test images that are robust against all three attacks simultaneously.

# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [1]:
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

Downloading...
From: https://drive.google.com/uc?id=17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes
To: /content/linf_base.pt
100% 11.2M/11.2M [00:00<00:00, 53.7MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=afb83f4e-d06d-4bdb-80e2-71cc30a16a87
To: /content/cifar10_train_10k.pt
100% 30.8M/30.8M [00:00<00:00, 65.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /content/cifar10_test_1k.pt
100% 3.08M/3.08M [00:00<00:00, 9.86MB/s]


In [2]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [3]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [4]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [5]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    # TODO: implement L-infinity PGD (use input_grad and clip_box)
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.empty_like(x).uniform_(-eps, eps)
        delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad.sign()
        delta = delta.clamp(-eps, eps)
        delta = clip_box(x, delta)
    return (x + delta).detach()


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    # TODO: implement L2 PGD (use input_grad, l2_norm and clip_box)
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.randn_like(x)
        delta_norm = l2_norm(delta).clamp(min=1e-12)
        delta = delta / delta_norm * torch.rand(x.shape[0], 1, 1, 1, device=x.device) * eps
        delta = clip_box(x, delta)

    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        grad_norm = l2_norm(grad).clamp(min=1e-12)
        delta = delta + alpha * grad / grad_norm
        delta_norm = l2_norm(delta).clamp(min=1e-12)
        delta = delta * (eps / delta_norm).clamp(max=1.0)
        delta = clip_box(x, delta)
    return (x + delta).detach()

In [6]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
        delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
    delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [7]:
def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

In [8]:
# The base model: L-infinity PGD adversarial training for 30 epochs on all 50,000 training images.
results_base = test_model(load_base_model())

clean 78.7  linf 48.4  l2 59.8  l1 47.1  union 42.6   (130 s)


# Adversarial Training

`adv_train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

If `gp_beta` is set, `adv_train` does the gradient projection of RAMP (part (b)) before the optimizer step.

In [9]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [10]:
def single_norm_loss(model, x, y, norm):
    # TODO: put the model in eval mode, attack with `norm`, put the model in train mode, and return the loss
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()
    return F.cross_entropy(model(x_adv), y)

In [16]:
def clean_loss(model, x, y):
    model.train()
    return F.cross_entropy(model(x), y)

model_standard = adv_train(load_base_model(), clean_loss, seed=0)
test_model(model_standard)

epoch 1: loss 0.522 (5 s)
epoch 2: loss 0.374 (10 s)
epoch 3: loss 0.319 (16 s)
epoch 4: loss 0.267 (21 s)
epoch 5: loss 0.217 (26 s)
epoch 6: loss 0.180 (32 s)
epoch 7: loss 0.151 (37 s)
epoch 8: loss 0.127 (43 s)
epoch 9: loss 0.112 (48 s)
epoch 10: loss 0.110 (54 s)
clean 87.3  linf 4.7  l2 16.8  l1 14.1  union 4.0   (136 s)


{'clean': 87.3, 'linf': 4.7, 'l2': 16.8, 'l1': 14.1, 'union': 4.0}

In [17]:
model_linf = adv_train(load_base_model(), single_norm_loss, seed=0,norm="linf")
results_linf = test_model(model_linf)

epoch 1: loss 1.203 (44 s)
epoch 2: loss 1.180 (88 s)
epoch 3: loss 1.152 (132 s)
epoch 4: loss 1.126 (176 s)
epoch 5: loss 1.097 (220 s)
epoch 6: loss 1.068 (264 s)
epoch 7: loss 1.049 (308 s)
epoch 8: loss 1.031 (352 s)
epoch 9: loss 1.013 (396 s)
epoch 10: loss 1.007 (439 s)
clean 78.6  linf 44.9  l2 57.2  l1 45.7  union 39.4   (133 s)


In [18]:
def fgsm_attack(model, x, y, epsilon, norm='linf'):
    #untargeted FGSM attack (one step of PGD)
    return ATTACKS[norm](model, x, y, epsilon, alpha = epsilon, steps=1, rand_init=False)
def evaluate_attack(model, model_name, epsilon_list = [2/255, 4/255, 8/255, 16/255]):
    model.eval()
    print(f"Evaluating attack for model: {model_name}")
    for epsilon in epsilon_list:
        fgsm_correct = 0
        pgd_correct = 0
        total = len(x_test)

        batch_size = 250
        for i in range(0, total, batch_size):
            x_batch = x_test[i:i+batch_size]
            y_batch = y_test[i:i+batch_size]
            x_adv = fgsm_attack(model, x_batch, y_batch, epsilon, norm='linf')
            with torch.no_grad():
                y_pred = model(x_adv)
                fgsm_correct += (y_pred.argmax(dim=1) == y_batch).sum().item()
            x_pgd = pgd_linf(model, x_batch, y_batch, epsilon, alpha = 2.5*epsilon/10, steps=10, rand_init=True)
            with torch.no_grad():
                y_pred = model(x_pgd)
                pgd_correct += (y_pred.argmax(dim=1) == y_batch).sum().item()
        print(f"Epsilon = {epsilon*255:.1f}/255 | FGSM Accuracy: {100.0 * fgsm_correct/total:.2f}% | PGD Accuracy: {100.0 * pgd_correct/total:.2f}%")

In [15]:
evaluate_attack(model_standard, "Standard Model")
evaluate_attack(model_linf, "Linfinity Adversarial Model")

Evaluating attack for model: Standard Model
Epsilon = 2.0/255 | FGSM Accuracy: 26.50% | PGD Accuracy: 21.50%
Epsilon = 4.0/255 | FGSM Accuracy: 11.50% | PGD Accuracy: 5.00%
Epsilon = 8.0/255 | FGSM Accuracy: 2.60% | PGD Accuracy: 0.00%
Epsilon = 16.0/255 | FGSM Accuracy: 0.90% | PGD Accuracy: 0.00%
Evaluating attack for model: Linfinity Adversarial Model
Epsilon = 2.0/255 | FGSM Accuracy: 33.10% | PGD Accuracy: 33.10%
Epsilon = 4.0/255 | FGSM Accuracy: 29.20% | PGD Accuracy: 29.00%
Epsilon = 8.0/255 | FGSM Accuracy: 24.20% | PGD Accuracy: 24.40%
Epsilon = 16.0/255 | FGSM Accuracy: 15.30% | PGD Accuracy: 12.80%


In [19]:
model_l2 = adv_train(load_base_model(), single_norm_loss, norm="l2")
results_l2 = test_model(model_l2)

epoch 1: loss 0.897 (44 s)
epoch 2: loss 0.826 (88 s)
epoch 3: loss 0.787 (132 s)
epoch 4: loss 0.754 (177 s)
epoch 5: loss 0.715 (221 s)
epoch 6: loss 0.674 (265 s)
epoch 7: loss 0.632 (308 s)
epoch 8: loss 0.610 (352 s)
epoch 9: loss 0.591 (396 s)
epoch 10: loss 0.584 (440 s)
clean 82.8  linf 32.1  l2 59.8  l1 53.9  union 31.9   (134 s)


In [20]:
model_l1 = adv_train(load_base_model(), single_norm_loss, norm="l1")
results_l1 = test_model(model_l1)

epoch 1: loss 1.231 (44 s)
epoch 2: loss 1.081 (88 s)
epoch 3: loss 0.997 (133 s)
epoch 4: loss 0.902 (177 s)
epoch 5: loss 0.840 (221 s)
epoch 6: loss 0.786 (265 s)
epoch 7: loss 0.684 (309 s)
epoch 8: loss 0.601 (353 s)
epoch 9: loss 0.507 (397 s)
epoch 10: loss 0.482 (441 s)
clean 81.7  linf 5.0  l2 31.7  l1 62.6  union 5.0   (134 s)


# Part (b): Multi-Norm Adversarial Training (RAMP)

* `ramp_loss`: attack with $q = \ell_\infty$ and $r = \ell_1$. The loss is the mean of
  $\max(\mathcal{L}_{CE}(x^q), \mathcal{L}_{CE}(x^r))$ plus `lam` times $\mathcal{L}_{KL} = KL(p_q \| p_r)$.
  Calculate $\mathcal{L}_{KL}$ only on the images that the model classifies correctly under the $\ell_q$ attack.
* `gradient_projection(g_n, g_a)`: $\cos(g_n, g_a) \cdot g_n$ if $\cos(g_n, g_a) > 0$, else $0$.

`adv_train(..., gp_beta=0.5)` then uses `p.grad = gp_beta * GP(g_n, g_a) + (1 - gp_beta) * g_a` for each parameter
tensor, where $g_a$ is the gradient of `ramp_loss` and $g_n$ is the gradient of the cross-entropy loss on the clean images.

In [21]:
def ramp_loss(model, x, y, lam=0.5):
    # TODO: in eval mode, attack with linf and l1; in train mode, return the max of the two losses plus lam * KL
    model.eval()
    x_adv_linf = attack(model, x, y, "linf")
    x_adv_l1 = attack(model, x, y, "l1")
    model.train()
    model_linf = model(x_adv_linf)
    model_l1 = model(x_adv_l1)

    # Compute the losses for each norm
    loss_linf = F.cross_entropy(model_linf, y)
    loss_l1 = F.cross_entropy(model_l1, y)

    # Return the maximum loss plus the KL divergence term
    max_loss = torch.max(loss_linf, loss_l1).mean()
    #calculate only on images correctly classified under lq attack
    correct_linf = model_linf.argmax(dim=1) == y

    if correct_linf.any():
        kl_div = F.kl_div(F.log_softmax(model_l1[correct_linf], dim=1), F.softmax(model_linf[correct_linf], dim=1), reduction='batchmean')
    else:
        kl_div = torch.tensor(0.0, device=x.device)

    return max_loss + lam * kl_div

def gradient_projection(g_n, g_a):
    # TODO: GP(g_n, g_a) for one parameter tensor

    g_n_flat = g_n.flatten()
    g_a_flat = g_a.flatten()

    dot_product = torch.dot(g_n_flat, g_a_flat)
    norm_g_n = torch.norm(g_n_flat)
    norm_g_a = torch.norm(g_a_flat)

    epsilon = 1e-12  # Small value to avoid division by zero
    cos_theta = dot_product / (norm_g_n * norm_g_a + epsilon)

    if cos_theta > 0 and not torch.isnan(cos_theta):
        return cos_theta * g_n
    else:
        return torch.zeros_like(g_n)

In [22]:
model_ramp = adv_train(load_base_model(), ramp_loss, gp_beta=0.5)
test_model(model_ramp)

epoch 1: loss 1.347 (95 s)
epoch 2: loss 1.269 (190 s)
epoch 3: loss 1.245 (284 s)
epoch 4: loss 1.225 (378 s)
epoch 5: loss 1.198 (472 s)
epoch 6: loss 1.173 (566 s)
epoch 7: loss 1.145 (660 s)
epoch 8: loss 1.132 (755 s)
epoch 9: loss 1.117 (848 s)
epoch 10: loss 1.113 (942 s)
clean 79.7  linf 41.2  l2 62.2  l1 54.5  union 40.7   (133 s)


{'clean': 79.7, 'linf': 41.2, 'l2': 62.2, 'l1': 54.5, 'union': 40.7}

# Part (c): Your Method

Write the training loss of your method in `my_loss`, and train with `adv_train(load_base_model(), my_loss)`.
If your method needs a different update (not the gradient of one loss), write your own copy of `adv_train`.
Run your method and RAMP with `seed=0` and `seed=1`.
Then evaluate your best model again with stronger attacks: `test_model(model, steps=200, restarts=5)`.

In [23]:
def my_loss(model, x, y, lam=0.5):
    # TODO: your method

    model.eval()
    x_adv_linf = attack(model, x, y, "linf")
    x_adv_l1 = attack(model, x, y, "l1")
    x_adv_l2 = attack(model, x, y, "l2")

    model.train()
    model_linf = model(x_adv_linf)
    model_l1 = model(x_adv_l1)
    model_l2 = model(x_adv_l2)

    loss_linf = F.cross_entropy(model_linf, y, reduction='none')
    loss_l1 = F.cross_entropy(model_l1, y, reduction='none')
    loss_l2 = F.cross_entropy(model_l2, y, reduction='none')

    losses = torch.stack([loss_linf.mean(), loss_l1.mean(), loss_l2.mean()], dim=0)

    weights = F.softmax(losses, dim=0)

    # Compute the weighted sum of the losses across norms
    weighted_loss = (weights[0] * loss_linf + weights[1] * loss_l1 + weights[2] * loss_l2)
    max_loss = torch.stack([loss_linf, loss_l1, loss_l2], dim=1).max(dim=1)[0]
    return (weighted_loss + lam * max_loss).mean()

In [24]:
model_new_seed0 = adv_train(load_base_model(), my_loss, seed=0)
result_seed0 = test_model(model_new_seed0)

model_new_seed1 = adv_train(load_base_model(), my_loss, seed=1)
result_seed1 = test_model(model_new_seed1)

epoch 1: loss 1.928 (133 s)
epoch 2: loss 1.824 (265 s)
epoch 3: loss 1.759 (396 s)
epoch 4: loss 1.707 (527 s)
epoch 5: loss 1.660 (659 s)
epoch 6: loss 1.592 (791 s)
epoch 7: loss 1.550 (924 s)
epoch 8: loss 1.507 (1056 s)
epoch 9: loss 1.475 (1188 s)
epoch 10: loss 1.461 (1321 s)
clean 78.6  linf 41.2  l2 62.2  l1 57.4  union 41.1   (133 s)
epoch 1: loss 1.930 (132 s)
epoch 2: loss 1.812 (264 s)
epoch 3: loss 1.760 (395 s)
epoch 4: loss 1.701 (527 s)
epoch 5: loss 1.650 (660 s)
epoch 6: loss 1.593 (791 s)
epoch 7: loss 1.542 (922 s)
epoch 8: loss 1.493 (1055 s)
epoch 9: loss 1.470 (1187 s)
epoch 10: loss 1.458 (1319 s)
clean 77.6  linf 41.9  l2 62.3  l1 57.3  union 41.7   (133 s)


In [25]:
model_ramp_seed0 = adv_train(load_base_model(), ramp_loss, gp_beta=0.5, seed=0)
result_ramp_seed0 = test_model(model_ramp_seed0)

model_ramp_seed1 = adv_train(load_base_model(), ramp_loss, gp_beta=0.5, seed=1)
result_ramp_seed1 = test_model(model_ramp_seed1)

epoch 1: loss 1.347 (94 s)
epoch 2: loss 1.268 (189 s)
epoch 3: loss 1.244 (283 s)
epoch 4: loss 1.226 (377 s)
epoch 5: loss 1.198 (472 s)
epoch 6: loss 1.173 (566 s)
epoch 7: loss 1.145 (660 s)
epoch 8: loss 1.132 (754 s)
epoch 9: loss 1.117 (848 s)
epoch 10: loss 1.113 (943 s)
clean 79.6  linf 40.8  l2 61.8  l1 54.5  union 40.4   (133 s)
epoch 1: loss 1.348 (94 s)
epoch 2: loss 1.272 (188 s)
epoch 3: loss 1.243 (283 s)
epoch 4: loss 1.217 (377 s)
epoch 5: loss 1.197 (472 s)
epoch 6: loss 1.169 (566 s)
epoch 7: loss 1.148 (660 s)
epoch 8: loss 1.135 (754 s)
epoch 9: loss 1.126 (848 s)
epoch 10: loss 1.117 (943 s)
clean 79.8  linf 42.4  l2 62.7  l1 55.2  union 41.8   (134 s)
